# 001 - Gather Raw Anime Source Data

This notebook is the raw-first replacement for trying to build the perfect processed dataset in one pass.

The goal of this step is not to decide the final catalog labels. The goal is to keep source payloads:

- MAL/Jikan full anime payloads for valid anime types and all statuses, including `Not yet aired`.
- Jikan recommendation payloads for titles that are already released or airing.
- Jikan character payloads with compact Japanese voice-actor data.
- AniList media payloads immediately after each accepted MAL id, including compact character and Japanese voice-actor edges.
- AniDB/Shoko cache mirrored as a fallback source, mostly for episode/runtime/studio/demographic gaps.

The raw source index stores `mal_id`, `anilist_id`, and `anidb_id` together so later steps can build, compare, and repair cleanly.


In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

SCRIPT = BASE_DIR / "src" / "01_gather_raw_sources.py"
SUMMARY = BASE_DIR / "data" / "build" / "raw_dataset_gather_summary.json"
CHECKPOINT = BASE_DIR / "data" / "build" / "raw_dataset_gather_checkpoint.json"
FAILED = BASE_DIR / "data" / "build" / "raw_dataset_failed_api_requests.json"
INVALID = BASE_DIR / "data" / "build" / "raw_dataset_invalid_ids.json"
INDEX = BASE_DIR / "data" / "raw_sources" / "raw_source_index.csv"

print("Project:", BASE_DIR)
print("Gather script:", SCRIPT)



def run_streaming(command, cwd=None):
    """Run a script and print stdout/stderr line-by-line while it is still running."""
    cwd = cwd or globals().get("BASE_DIR") or globals().get("ROOT") or Path.cwd()
    command = [str(part) for part in command]
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)
    return return_code


Project: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect
Gather script: C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\01_gather_raw_sources.py


## Optional Jikan Character / Voice-Actor Backfill

Use this if the anime payloads are already gathered but some compact character/Japanese voice-actor rows are missing or failed. The command only fetches pending character payloads by default, saves each successful MAL id into `data/raw_sources/mal_jikan/jikan_character_voice_actor_cache.json`, and records retryable failures in `data/build/raw_dataset_failed_api_requests.json`.

In [2]:
RUN_JIKAN_CHARACTER_BACKFILL = True
JIKAN_CHARACTER_BACKFILL_LIMIT = None  # use None for all pending character rows

if RUN_JIKAN_CHARACTER_BACKFILL:
    command = [
        sys.executable,
        str(SCRIPT),
        "--jikan-characters",
        "--retry-failed",
    ]
    if JIKAN_CHARACTER_BACKFILL_LIMIT is not None:
        command.extend(["--limit", str(JIKAN_CHARACTER_BACKFILL_LIMIT)])
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=BASE_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
else:
    print("Jikan character/VA backfill skipped. Set RUN_JIKAN_CHARACTER_BACKFILL = True to run it.")


Running: c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\python.exe C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\01_gather_raw_sources.py --jikan-characters --retry-failed
[2026-07-11T02:42:33] JIKAN_CHARACTERS_BACKFILL_START | accepted=17274 | pending=0 | selected=0 | refresh_existing=False
[2026-07-11T02:42:38] RAW_GATHER_COMPLETE | {"updated_at": "2026-07-11T02:42:31", "jikan_characters": {"accepted_mal_ids": 17274, "pending_character_ids": 0, "selected_character_ids": 0, "jikan_character_live_requests": 0, "jikan_character_failed": 0, "jikan_character_cached_missing": 0, "jikan_character_cache_entries": 17297, "jikan_character_cache_file": "C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources\\mal_jikan\\jikan_character_voice_actor_cache.json"}, "raw_source_index_rows": 17274, "raw_source_index_csv": "C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources\\raw_source_index.csv", "raw_cac

## Test Run

Use a small limit before a long collection. This hits the same code path as the full run: MAL candidate ids are downloaded, Jikan is called, every accepted row immediately gets an AniList lookup, and the AniDB cache is mirrored if requested.

Keep `RUN_TEST = False` after the smoke test so the notebook is safe to reopen without making live API calls.


In [ ]:
RUN_TEST = False
if RUN_TEST:
    run_streaming([sys.executable, str(SCRIPT), "--jikan", "--anidb", "--limit", "20", "--retry-failed"])


## Full/Resume Raw Gathering

This is the main raw collection step from zero. It intentionally gathers source payloads first and delays final dataset decisions until `002` and `003`.

What this cell does:

1. Optionally downloads the Shoko `Anime_HTTP.zip`, imports the XML files into the AniDB fallback cache, and deletes the disposable zip.
2. Downloads the current MAL candidate-id cache.
3. Fetches each valid Jikan anime payload.
4. When a MAL id passes the type/status/score rule, fetches the matching AniList payload immediately.
5. Fetches compact Jikan character and Japanese voice-actor payloads for accepted titles.
6. Fetches Jikan recommendations only for titles that are already released or airing.
7. Mirrors the AniDB cache into `data/raw_sources/anidb/` for the raw-source build.

The script writes progress to the notebook output and to `logs/raw_dataset_gather_log.txt`, while failures and invalid ids are kept under `data/build/`.


In [3]:
RUN_FULL_GATHER = True
REFRESH_SHOKO_ANIDB_CACHE = False
FETCH_JIKAN_CHARACTERS = True


def run_streaming(command):
    print("Running:", " ".join(str(part) for part in command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=BASE_DIR,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


if RUN_FULL_GATHER:
    command = [sys.executable, str(SCRIPT), "--jikan", "--anidb", "--retry-failed"]
    if not FETCH_JIKAN_CHARACTERS:
        command.append("--skip-characters")
    if REFRESH_SHOKO_ANIDB_CACHE:
        command.append("--refresh-shoko")
    run_streaming(command)

    if CHECKPOINT.exists():
        print("\nCurrent checkpoint")
        display(json.loads(CHECKPOINT.read_text(encoding="utf-8")))

    if SUMMARY.exists():
        print("\nRaw gather summary")
        display(json.loads(SUMMARY.read_text(encoding="utf-8")))

    if FAILED.exists():
        failed = json.loads(FAILED.read_text(encoding="utf-8"))
        failed_items = list((failed.get("items") or {}).values())
        if failed_items:
            print("\nRecent failed API requests")
            display(pd.DataFrame(failed_items).tail(25))

    if INVALID.exists():
        invalid = json.loads(INVALID.read_text(encoding="utf-8"))
        invalid_items = list((invalid.get("items") or {}).values())
        if invalid_items:
            print("\nRecent invalid candidates / skipped MAL ids")
            display(pd.DataFrame(invalid_items).tail(25))


Running: c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\python.exe C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\01_gather_raw_sources.py --jikan --anidb --retry-failed
[2026-07-11T02:42:48] DOWNLOAD_START | https://raw.githubusercontent.com/purarue/mal-id-cache/master/cache/anime_cache.json -> C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\mal_candidate_ids.json
[2026-07-11T02:42:49] DOWNLOAD_COMPLETE | C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\mal_candidate_ids.json
[2026-07-11T02:42:49] MAL_CANDIDATE_IDS_READY | C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\data\raw\mal_candidate_ids.json
[2026-07-11T02:42:51] CHECKPOINT | 25/30204 | last_mal_id=43 | accepted=25 | failures=0
[2026-07-11T02:42:51] CHECKPOINT | 50/30204 | last_mal_id=68 | accepted=50 | failures=0
[2026-07-11T02:42:51] CHECKPOINT | 75/30204 | last_mal_id=95 | accepted=75 | failures=0
[2026-07-11T02:42:51] CHECKPOINT | 100

{'updated_at': '2026-07-11T03:18:56',
 'stage': 'jikan_gather_complete',
 'total_candidates': 30204,
 'accepted_seen': 17275,
 'jikan_anime_live_requests': 4,
 'jikan_character_live_requests': 0,
 'anilist_live_requests': 4,
 'jikan_recommendation_live_requests': 0,
 'jikan_anime_cache_entries': 30190,
 'jikan_character_cache_entries': 17297,
 'anilist_cache_entries': 17278,
 'jikan_recommendation_cache_entries': 16743,
 'failure_count': 21}


Raw gather summary


{'updated_at': '2026-07-11T02:42:48',
 'jikan': {'candidate_ids': 30204,
  'accepted_seen': 17275,
  'jikan_anime_live_requests': 4,
  'jikan_character_live_requests': 0,
  'anilist_live_requests': 4,
  'jikan_recommendation_live_requests': 0,
  'jikan_anime_cache_entries': 30190,
  'jikan_character_cache_entries': 17297,
  'anilist_cache_entries': 17278,
  'jikan_recommendation_cache_entries': 16743},
 'anidb': {'anidb_cache_entries': 15842,
  'anidb_cache_file': 'C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources\\anidb\\anidb_metadata_cache.json'},
 'raw_source_index_rows': 17278,
 'raw_source_index_csv': 'C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources\\raw_source_index.csv',
 'raw_cache_files': {'jikan_anime': 'C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources\\mal_jikan\\jikan_anime_full_cache.json',
  'jikan_characters': 'C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 202


Recent failed API requests


,mal_id,stage,error,retryable,last_attempt_at
0,64537,jikan_anime,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:06:44
1,64538,jikan_anime,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:07:18
2,64551,jikan_characters,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:08:04
3,64552,jikan_characters,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:08:49
4,64553,jikan_characters,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:09:34
5,64554,jikan_anime,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:10:09
6,64555,jikan_anime,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:10:43
7,64556,jikan_anime,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:11:16
8,64558,jikan_anime,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:11:50
9,64562,jikan_anime,"HTTP 504: {""status"":504,""type"":""BadResponseExc...",True,2026-07-11T03:12:23



Recent invalid candidates / skipped MAL ids


,mal_id,index,invalid_kind,type,status,score,reason,recorded_at
12887,64511,30161,filtered_candidate,Music,Finished Airing,NaN,invalid_type:Music,2026-07-11T03:06:10
12888,64512,30162,filtered_candidate,CM,Finished Airing,NaN,invalid_type:CM,2026-07-11T03:06:10
12889,64513,30163,filtered_candidate,Music,Finished Airing,NaN,invalid_type:Music,2026-07-11T03:06:10
12890,64521,30167,filtered_candidate,TV,Finished Airing,NaN,missing_score_non_not_yet_aired,2026-07-11T03:06:10
12891,64522,30168,filtered_candidate,PV,Finished Airing,NaN,invalid_type:PV,2026-07-11T03:06:10
12892,64525,30170,filtered_candidate,ONA,Finished Airing,NaN,missing_score_non_not_yet_aired,2026-07-11T03:06:11
12893,62430,29055,filtered_candidate,TV,Currently Airing,NaN,missing_score_non_not_yet_aired,2026-07-11T03:05:02
12894,62883,29357,filtered_candidate,TV,Currently Airing,NaN,missing_score_non_not_yet_aired,2026-07-11T03:05:21
12895,64533,30176,filtered_candidate,ONA,Finished Airing,NaN,missing_score_non_not_yet_aired,2026-07-11T03:06:11
12896,64539,30182,filtered_candidate,None,Not yet aired,NaN,invalid_type:None,2026-07-11T03:07:18


## AniList Catch-Up Only

Normally this is not needed because AniList is fetched right after each accepted Jikan row. Use this only if the MAL/Jikan pass finished but the AniList run was interrupted or rate-limited.


In [4]:
RUN_ANILIST_CATCH_UP = True
if RUN_ANILIST_CATCH_UP:
    run_streaming([sys.executable, str(SCRIPT), "--anilist"])

Running: c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\python.exe C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\01_gather_raw_sources.py --anilist
[2026-07-11T03:22:28] RAW_GATHER_COMPLETE | {"updated_at": "2026-07-11T03:22:17", "anilist": {"accepted_mal_ids": 17278, "anilist_missing_before_limit": 0, "anilist_live_updates": 0, "anilist_cache_entries": 17278}, "raw_source_index_rows": 17278, "raw_source_index_csv": "C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources\\raw_source_index.csv", "raw_cache_files": {"jikan_anime": "C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources\\mal_jikan\\jikan_anime_full_cache.json", "jikan_characters": "C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources\\mal_jikan\\jikan_character_voice_actor_cache.json", "jikan_recommendations": "C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\data\\raw_sources

## Seasonal / Targeted Refresh Pipeline

Use this when a new anime season starts or when a currently-airing title begins receiving scores. Summer 2026 is starting, so this wrapper first asks Jikan's season endpoints (`/seasons/now` and `/seasons/{year}/{season}`) for fresh seasonal MAL ids, refreshes stale currently-airing/current-season rows, can force specific MAL ids such as `59193`, rebuilds the processed dataset, and optionally rebuilds the VA/character/staff tables in the same run.

This is intentionally a wrapper around the normal scripts:

1. `01_gather_raw_sources.py` refreshes Jikan/AniList seasonal caches and targeted ids.
2. `02_build_anime_dataset.py` rebuilds `anime_dataset.csv/json` from raw caches.
3. `04_improve_anime_dataset.py` reapplies cleanup and rebuilds people/staff tables.

The command writes `data/build/seasonal_discovered_jikan_ids.csv`, `data/build/seasonal_refresh_candidates.csv`, and `data/build/seasonal_refresh_run_summary.json`. It streams progress, so you can see which phase is running instead of staring at a spinning cell.




In [2]:
RUN_SEASONAL_REFRESH_PIPELINE = True
SEASONAL_REFRESH_MAX_AGE_HOURS = 1
SEASONAL_REFRESH_DATE = None  # None uses today; example: "2026-07-06"
INCLUDE_PREVIOUS_SEASON = True  # Refresh Spring as Summer starts rolling out.

# Force-refresh specific MAL ids even if they are not already in the seasonal cache.
# These are forced even if discovery/candidate selection would otherwise skip them.
EXTRA_MAL_IDS = [59193, 60522, 63832, 63403, 62811, 62076]

REFRESH_CHARACTERS = True
RUN_RECENT_ANIDB = True      # Slow/courteous; enable only when you want fresh AniDB fallback too.
ANIDB_RECENT_LIMIT = None     # Optional safety limit for AniDB live calls.
REFRESH_TOP_FAVORITES = False # Very slow; only needed when refreshing top people/character favorite caches.
DROP_AUXILIARY_COLUMNS = False 

if RUN_SEASONAL_REFRESH_PIPELINE:
    cmd = [
        sys.executable,
        str(BASE_DIR / "src" / "12_run_seasonal_refresh.py"),
        "--max-age-hours", str(SEASONAL_REFRESH_MAX_AGE_HOURS),
    ]
    if SEASONAL_REFRESH_DATE:
        cmd.extend(["--date", SEASONAL_REFRESH_DATE])
    if INCLUDE_PREVIOUS_SEASON:
        cmd.append("--include-previous-season")
    else:
        cmd.append("--no-include-previous-season")
    if EXTRA_MAL_IDS:
        cmd.extend(["--ids", *[str(value) for value in EXTRA_MAL_IDS]])
    if REFRESH_CHARACTERS:
        cmd.append("--refresh-characters")
    else:
        cmd.append("--no-refresh-characters")
    if RUN_RECENT_ANIDB:
        cmd.append("--run-anidb-recent")
        if ANIDB_RECENT_LIMIT is not None:
            cmd.extend(["--anidb-limit", str(ANIDB_RECENT_LIMIT)])
    if REFRESH_TOP_FAVORITES:
        cmd.append("--refresh-top-favorites")
    if DROP_AUXILIARY_COLUMNS:
        cmd.append("--drop-auxiliary-columns")
    run_streaming(cmd)
else:
    print("Seasonal refresh pipeline skipped.")

summary_path = BASE_DIR / "data" / "build" / "seasonal_refresh_run_summary.json"
candidate_csv = BASE_DIR / "data" / "build" / "seasonal_refresh_candidates.csv"
discovery_csv = BASE_DIR / "data" / "build" / "seasonal_discovered_jikan_ids.csv"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    print(json.dumps(summary, indent=2, ensure_ascii=False))
if discovery_csv.exists():
    discovered = pd.read_csv(discovery_csv)
    print(f"Jikan seasonal discovery rows: {len(discovered):,}")
    display(discovered.head(50))
if candidate_csv.exists():
    seasonal_candidates = pd.read_csv(candidate_csv)
    print(f"Cache-based seasonal refresh candidate rows: {len(seasonal_candidates):,}")
    display(seasonal_candidates.head(50))




Running: c:\Users\CHAMPUX\AppData\Local\Programs\Python\Python313\python.exe C:\Users\CHAMPUX\Downloads\UPC TRABAJOS 2026\BIG DATA\proyect\src\12_run_seasonal_refresh.py --max-age-hours 1 --include-previous-season --ids 59193 60522 63832 63403 62811 62076 --refresh-characters --run-anidb-recent

=== Seasonal Jikan/AniList refresh ===
Running: python src\01_gather_raw_sources.py --seasonal-refresh --seasonal-refresh-max-age-hours 1.0 --skip-mal-id-download --jikan-sleep 1.0 --anilist-sleep 2.2 --include-previous-season --refresh-characters
[2026-07-11T16:46:42] JIKAN_REQUEST_EXCEPTION | /seasons/2026/spring?page=1 | ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')) | sleep=2.0s
[2026-07-11T16:46:48] JIKAN_REQUEST_EXCEPTION | /seasons/2026/spring?page=1 | ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')) | sleep=4.0s
[2026-07-11T16:46:56] JIKAN_REQUEST_EXCEPTION | /seas

CalledProcessError: Command '['c:\\Users\\CHAMPUX\\AppData\\Local\\Programs\\Python\\Python313\\python.exe', 'C:\\Users\\CHAMPUX\\Downloads\\UPC TRABAJOS 2026\\BIG DATA\\proyect\\src\\12_run_seasonal_refresh.py', '--max-age-hours', '1', '--include-previous-season', '--ids', '59193', '60522', '63832', '63403', '62811', '62076', '--refresh-characters', '--run-anidb-recent']' returned non-zero exit status 1.

## Current Raw Gathering State

These tables are the quick health check: how many raw rows are aligned, which ids failed, and which ids were excluded or invalid before the processed catalog is built.


In [ ]:
if SUMMARY.exists():
    display(json.loads(SUMMARY.read_text(encoding="utf-8")))

if INDEX.exists():
    raw_index = pd.read_csv(INDEX)
    display(raw_index.head())
    display(raw_index[["mal_id", "anilist_id", "anidb_id"]].notna().mean().rename("coverage").to_frame())

if FAILED.exists():
    failed = json.loads(FAILED.read_text(encoding="utf-8"))
    failed_items = list((failed.get("items") or {}).values())
    display(pd.DataFrame(failed_items).head(25))

if INVALID.exists():
    invalid = json.loads(INVALID.read_text(encoding="utf-8"))
    invalid_items = list((invalid.get("items") or {}).values())
    display(pd.DataFrame(invalid_items).head(25))


## Optional AniDB Recent Live Backfill

The Shoko AniDB XML cache is dated around `2025-09-14`, so newer anime can have a MAL AniDB id but no usable AniDB metadata in our cache. This optional step targets only accepted MAL entries airing on or after that date, prioritizes `Rx - Hentai` entries first, then MAL popularity, and updates `data/raw_sources/anidb/anidb_metadata_cache.json` as each successful AniDB HTTP response arrives.

Run this with a small limit first. If AniDB returns a ban/rate-limit response, the script stops and keeps the successful cache updates already saved.

In [1]:
import importlib.util
import sys

for import_path in (BASE_DIR, BASE_DIR / "src"):
    import_path_text = str(import_path)
    if import_path_text not in sys.path:
        sys.path.insert(0, import_path_text)

spec = importlib.util.spec_from_file_location("raw_gather", SCRIPT)
raw_gather = importlib.util.module_from_spec(spec)
spec.loader.exec_module(raw_gather)

anidb_candidates = raw_gather.recent_anidb_live_candidates("2025-09-14")
source_cache = raw_gather.load_cache(raw_gather.ANIDB_SOURCE_CACHE_FILE)
raw_cache = raw_gather.load_cache(raw_gather.ANIDB_RAW_CACHE_FILE)
anidb_cache_items = {}
anidb_cache_items.update(source_cache.get("items", {}))
anidb_cache_items.update(raw_cache.get("items", {}))
pending_anidb_candidates = [
    row for row in anidb_candidates
    if (anidb_cache_items.get(str(row["anidb_id"])) or {}).get("source") != "live_http"
]
print(f"Recent AniDB live candidates: {len(anidb_candidates):,}")
print(f"Remaining not-live candidates: {len(pending_anidb_candidates):,}")
display(pd.DataFrame(pending_anidb_candidates).head(30))

NameError: name 'BASE_DIR' is not defined

In [ ]:
RUN_ANIDB_RECENT_LIVE_BACKFILL = True
ANIDB_RECENT_LIMIT = None  # use None for the full prioritized queue
ANIDB_RECENT_SINCE = "2025-09-14"

if RUN_ANIDB_RECENT_LIVE_BACKFILL:
    command = [
        sys.executable,
        str(SCRIPT),
        "--anidb-live-recent",
        "--anidb",
        "--anidb-since",
        ANIDB_RECENT_SINCE,
    ]
    if ANIDB_RECENT_LIMIT is not None:
        command.extend(["--limit", str(ANIDB_RECENT_LIMIT)])
    print("Running:", " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=BASE_DIR,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
else:
    print("AniDB recent live backfill skipped. Set RUN_ANIDB_RECENT_LIVE_BACKFILL = True to run it.")